In [9]:
import torch

In [10]:
import math

## Adam Optimizer

### Algorithm (per parameter)
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

In [11]:

class Adam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * p.grad
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * p.grad ** 2
                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)
                p -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [12]:

w = torch.randn(4, 3, requires_grad=True)
opt = Adam([w], lr=0.01)
for i in range(20):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'Step {i}: loss={loss.item():.4f}')

Step 0: loss=7.9808
Step 1: loss=7.8361
Step 2: loss=7.6939
Step 3: loss=7.5543
Step 4: loss=7.4171
Step 5: loss=7.2825
Step 6: loss=7.1505
Step 7: loss=7.0210
Step 8: loss=6.8941
Step 9: loss=6.7697
Step 10: loss=6.6478
Step 11: loss=6.5283
Step 12: loss=6.4111
Step 13: loss=6.2962
Step 14: loss=6.1836
Step 15: loss=6.0731
Step 16: loss=5.9648
Step 17: loss=5.8585
Step 18: loss=5.7542
Step 19: loss=5.6519


## Cosine LR Scheduler with Warmup

### Schedule
```
step < warmup:  lr = max_lr * step / warmup_steps  (linear ramp)
step >= warmup: lr = min_lr + 0.5*(max_lr-min_lr)*(1 + cos(π * progress))
```
where `progress = (step - warmup) / (total - warmup)`

In [13]:
def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step >= total_steps:
        return min_lr
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

In [15]:
lrs = [cosine_lr_schedule(i, 100, 10, 0.001) for i in range(101)]
print(f'Start: {lrs[0]:.6f}, Warmup end: {lrs[10]:.6f}, Mid: {lrs[55]:.6f}, End: {lrs[100]:.6f}')

Start: 0.000000, Warmup end: 0.001000, Mid: 0.000500, End: 0.000000
